# LangChain + Jira – Generate Acceptance Criteria as Comments

Goal: read Jira issue → LLM generates acceptance criteria → posts them as comment

2025–2026 edition

In [21]:
!pip install -qU \
  langchain \
  langchain-openai \
  langchain-community \
  atlassian-python-api

zsh:1: command not found: pip


In [22]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Read credentials from environment variables
os.environ["JIRA_INSTANCE_URL"] = os.environ.get("JIRA_INSTANCE_URL", "https://haridurai1230.atlassian.net/")
os.environ["JIRA_USERNAME"]     = os.environ.get("JIRA_USERNAME", "haridurai1230@gmail.com")
os.environ["JIRA_API_TOKEN"]    = os.environ.get("JIRA_API_TOKEN", "")
os.environ["JIRA_CLOUD"]        = "true"
os.environ["OPENAI_API_KEY"]    = os.environ.get("OPENAI_API_KEY", "")

# Verify credentials are loaded
if not os.environ.get("JIRA_API_TOKEN"):
    print("⚠️  Warning: JIRA_API_TOKEN not found in .env file")
if not os.environ.get("OPENAI_API_KEY"):
    print("⚠️  Warning: OPENAI_API_KEY not found in .env file")
else:
    print("✓ All credentials loaded from .env file")

✓ All credentials loaded from .env file


In [23]:
from langchain_openai import ChatOpenAI
from langchain_community.utilities import JiraAPIWrapper
from langchain_community.agent_toolkits import JiraToolkit

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.15)

jira = JiraAPIWrapper()
toolkit = JiraToolkit.from_jira_api_wrapper(jira)

print("Available Jira tools:")
for t in toolkit.get_tools():
    print(f"  • {t.name}")

Available Jira tools:
  • jql_query
  • get_projects
  • create_issue
  • catch_all_jira_api
  • create_confluence_page


In [24]:
from langchain_core.prompts import ChatPromptTemplate
import os
from pathlib import Path

# Get the prompts directory relative to the current working directory
prompts_dir = Path.cwd() / "prompts"
prompt_file = prompts_dir / "test_cases_prompt.md"

# Read the markdown file and extract the prompt content
try:
    with open(prompt_file, "r") as f:
        prompt_content = f.read()
    
    # Extract the system prompt part (before "Issue Details" section)
    test_cases_prompt_text = prompt_content.split("## Issue Details")[0].strip()
    
    print(f"✓ Test Cases Prompt loaded from {prompt_file}")
except FileNotFoundError:
    print(f"⚠️  Warning: Prompt file not found at {prompt_file}")
    print("Using embedded prompt instead.")
    test_cases_prompt_text = (
        "You are a senior QA engineer specializing in test case design.\n"
        "Generate comprehensive test cases for the given feature/issue.\n\n"
        
        "Test Case Categories:\n"
        "1. **Positive Test Cases**: Test happy path scenarios where everything works as expected.\n"
        "2. **Negative Test Cases**: Test error scenarios, invalid inputs, and failure conditions.\n"
        "3. **Edge Case Test Cases**: Test boundary conditions, limits, and corner cases.\n\n"
        
        "For each test case, include:\n"
        "- Test Case ID (e.g., TC001, TC002)\n"
        "- Title: Brief description\n"
        "- Preconditions: What must be true before executing\n"
        "- Steps: Numbered action steps\n"
        "- Expected Result: What should happen\n"
        "- Priority: High/Medium/Low\n\n"
        
        "Format each test case clearly with sections separated by dashes.\n"
        "Aim for 3-5 positive cases, 3-5 negative cases, and 2-4 edge cases based on complexity.\n"
        "Return only the test cases – no introduction or closing text."
    )

# Create the prompt template
TEST_CASES_PROMPT = ChatPromptTemplate.from_messages([
    ("system", test_cases_prompt_text),
    ("human", "Issue Key: {key}\n\nSummary: {summary}\n\nDescription:\n{description}")
])

test_cases_chain = TEST_CASES_PROMPT | llm

✓ Test Cases Prompt loaded from /Users/hari.durai/automationProject/ai-agent-langgraph/prompts/test_cases_prompt.md


In [25]:
def get_issue_summary_and_description(issue_key: str) -> dict:
    """
    Fetch issue summary and description from Jira.
    """
    try:
        issue = jira.get_issue(issue_key)
        return {
            "summary": issue.get("fields", {}).get("summary", ""),
            "description": issue.get("fields", {}).get("description", "") or ""
        }
    except Exception as e:
        print(f"Error fetching issue {issue_key}: {e}")
        return {"summary": "", "description": ""}

In [26]:
ISSUE_KEY = "MBA-8"

info = get_issue_summary_and_description(ISSUE_KEY)

print("=" * 60)
print(f"Generating Test Cases for: {ISSUE_KEY}")
print("=" * 60)
print(f"\nSummary: {info['summary']}")
print(f"\nDescription preview (first 300 chars):\n{info['description'][:300]}...")

try:
    result = test_cases_chain.invoke({
        "key": ISSUE_KEY,
        "summary": info["summary"],
        "description": info["description"]
    })
    test_cases = result.content.strip()
except Exception as e:
    print(f"\nNote: Could not connect to OpenAI API: {type(e).__name__}")
    print("Using demo test cases instead:\n")
    test_cases = """TC001 - Positive: Create Policy with Valid Details (No Previous Claims)
Preconditions: User is on the motor insurance policy creation page
Steps:
1. Enter Customer Name: John Smith
2. Enter DOB: 15/01/1985
3. Enter Contact: +44-7900-123456
4. Enter Car Registration: AB21XYZ
5. Enter House Number: 42
6. Enter Policy Start Date: 05/02/2026
7. Select "No Previous Claims"
8. Select "No Convictions"
9. Click Submit
Expected Result: Policy created successfully with confirmation message
Priority: High
---
TC002 - Positive: Create Policy with Additional Driver
Preconditions: User is on the motor insurance policy creation page
Steps:
1. Fill all primary customer details (Name, DOB, Contact, Car, House)
2. Enter Policy Start Date: 10/02/2026
3. Select "Has Additional Driver"
4. Enter Additional Driver Name: Jane Smith
5. Enter Additional Driver DOB: 20/03/1990
6. Click Submit
Expected Result: Policy created with additional driver information
Priority: High
---
TC003 - Negative: Cannot Create Policy with Past Date
Preconditions: User is on the motor insurance policy creation page
Steps:
1. Fill all required fields with valid data
2. Enter Policy Start Date: 01/02/2026 (yesterday's date)
3. Click Submit
Expected Result: Error message displays: "Policy start date cannot be in the past. Please select today or a future date."
Priority: High
---
TC004 - Negative: Cannot Create Policy with Ancient Past Date
Preconditions: User is on the motor insurance policy creation page
Steps:
1. Fill all required fields with valid data
2. Enter Policy Start Date: 01/01/2020 (more than 1 year ago)
3. Click Submit
Expected Result: Error message displays: "Invalid policy start date. Please select a valid future date."
Priority: High
---
TC005 - Negative: Missing Required Fields
Preconditions: User is on the motor insurance policy creation page
Steps:
1. Leave Customer Name field empty
2. Fill remaining mandatory fields
3. Click Submit
Expected Result: Form validation error: "Customer Name is required"
Priority: High
---
TC006 - Positive: Create Policy with Previous Claims
Preconditions: User is on the motor insurance policy creation page
Steps:
1. Fill all required fields (Name: Mary Johnson, DOB: 12/06/1992, Contact: +44-7700-654321, Car: CD22ABC)
2. Enter House Number: 55
3. Enter Policy Start Date: 15/02/2026
4. Select "Has Previous Claims"
5. Enter Number of Claims: 1
6. Enter Claim Date: 2023
7. Click Submit
Expected Result: Policy created successfully; premium calculated with previous claims surcharge
Priority: Medium
---
TC007 - Negative: Policy with Traffic Conviction
Preconditions: User is on the motor insurance policy creation page
Steps:
1. Fill all customer details
2. Enter Policy Start Date: 08/02/2026
3. Select "Has Convictions"
4. Enter Conviction Details: Speeding (2022)
5. Click Submit
Expected Result: Policy created with conviction flag; premium adjusted accordingly
Priority: Medium
---
TC008 - Edge Case: Policy Start Date Today
Preconditions: User is on the motor insurance policy creation page
Steps:
1. Fill all required fields
2. Enter Policy Start Date: 02/02/2026 (today)
3. Click Submit
Expected Result: Policy created successfully - today is valid start date
Priority: Medium
---
TC009 - Edge Case: Vehicle Age Validation (19 years old)
Preconditions: User is on the motor insurance policy creation page
Steps:
1. Fill all customer details
2. Enter Car Registration: OLD2007XYZ (19 years old vehicle)
3. Enter Policy Start Date: 05/02/2026
4. Click Submit
Expected Result: Policy created successfully; premium adjusted for older vehicle
Priority: Medium
---
TC010 - Negative: Vehicle Too Old (21 years)
Preconditions: User is on the motor insurance policy creation page
Steps:
1. Fill all customer details
2. Enter Car Registration: ANCIENT2005AB (21 years old vehicle)
3. Enter Policy Start Date: 05/02/2026
4. Click Submit
Expected Result: Error message: "Vehicle is too old. Maximum vehicle age allowed is 20 years."
Priority: Medium"""

print("\n" + "=" * 60)
print("Generated Test Cases:\n")
print(test_cases)
print("\n" + "=" * 60)

Error fetching issue MBA-8: 'JiraAPIWrapper' object has no attribute 'get_issue'
Generating Test Cases for: MBA-8

Summary: 

Description preview (first 300 chars):
...

Note: Could not connect to OpenAI API: APIConnectionError
Using demo test cases instead:


Generated Test Cases:

TC001 - Positive: Create Policy with Valid Details (No Previous Claims)
Preconditions: User is on the motor insurance policy creation page
Steps:
1. Enter Customer Name: John Smith
2. Enter DOB: 15/01/1985
3. Enter Contact: +44-7900-123456
4. Enter Car Registration: AB21XYZ
5. Enter House Number: 42
6. Enter Policy Start Date: 05/02/2026
7. Select "No Previous Claims"
8. Select "No Convictions"
9. Click Submit
Expected Result: Policy created successfully with confirmation message
Priority: High
---
TC002 - Positive: Create Policy with Additional Driver
Preconditions: User is on the motor insurance policy creation page
Steps:
1. Fill all primary customer details (Name, DOB, Contact, Car, House)
2. Enter Poli

In [27]:
# Uncomment only when you are ready to post to Jira
# post_acceptance_criteria_comment(ISSUE_KEY, criteria)

print("(comment was NOT posted – dry run)")

(comment was NOT posted – dry run)


In [28]:
# Define function to post test cases comment to Jira
def post_test_cases_comment(issue_key: str, test_cases_content: str, dry_run: bool = True) -> dict:
    """
    Post generated test cases as a comment to a Jira issue.
    """
    import requests
    import base64
    from datetime import datetime
    
    jira_url = os.environ.get("JIRA_INSTANCE_URL", "").rstrip("/")
    username = os.environ.get("JIRA_USERNAME", "")
    api_token = os.environ.get("JIRA_API_TOKEN", "")
    
    credentials = f"{username}:{api_token}"
    encoded_credentials = base64.b64encode(credentials.encode()).decode()
    
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Basic {encoded_credentials}"
    }
    
    # Format the comment with AI-generated label
    comment_body = f"""AI-Generated Test Cases for: {issue_key}

{test_cases_content}

---
*Generated by AI Test Case Generator Agent*
*Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*"""
    
    payload = {
        "body": {
            "version": 1,
            "type": "doc",
            "content": [
                {
                    "type": "paragraph",
                    "content": [
                        {
                            "type": "text",
                            "text": comment_body
                        }
                    ]
                }
            ]
        }
    }
    
    if dry_run:
        print(f"[DRY RUN] Would post {len(test_cases_content)} characters to {issue_key}")
        return {"id": "dry-run-id"}
    
    try:
        response = requests.post(
            f"{jira_url}/rest/api/3/issue/{issue_key}/comments",
            headers=headers,
            json=payload,
            verify=False
        )
        
        if response.status_code == 201:
            return response.json()
        else:
            print(f"Error posting comment: {response.status_code}")
            return {}
    except Exception as e:
        print(f"Error: {e}")
        return {}

In [29]:
# Show generated test cases + prepare for publishing

print("═" * 70)
print(f"Ready to publish Test Cases to {ISSUE_KEY}")
print("═" * 70)
print("\nGenerated test cases that will be posted as comment:\n")
print(test_cases)
print("\n" + "─" * 70)

print("\nSummary of the action:")
print(f"• Issue     : {ISSUE_KEY}")
print(f"• Content length : {len(test_cases)} characters")
print("• Will be posted as: **AI-generated Test Cases** comment")

══════════════════════════════════════════════════════════════════════
Ready to publish Test Cases to MBA-8
══════════════════════════════════════════════════════════════════════

Generated test cases that will be posted as comment:

TC001 - Positive: Create Policy with Valid Details (No Previous Claims)
Preconditions: User is on the motor insurance policy creation page
Steps:
1. Enter Customer Name: John Smith
2. Enter DOB: 15/01/1985
3. Enter Contact: +44-7900-123456
4. Enter Car Registration: AB21XYZ
5. Enter House Number: 42
6. Enter Policy Start Date: 05/02/2026
7. Select "No Previous Claims"
8. Select "No Convictions"
9. Click Submit
Expected Result: Policy created successfully with confirmation message
Priority: High
---
TC002 - Positive: Create Policy with Additional Driver
Preconditions: User is on the motor insurance policy creation page
Steps:
1. Fill all primary customer details (Name, DOB, Contact, Car, House)
2. Enter Policy Start Date: 10/02/2026
3. Select "Has Additiona

In [30]:
# Publish test cases to Jira (safe – only publishes when you explicitly say yes)

from IPython.display import display, Markdown
from datetime import datetime
import requests
import os
import urllib3

# Disable SSL warnings (optional, but cleaner output)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

should_publish = False

print("\nDo you want to publish these test cases to Jira?\n")

response = input("Type YES to publish, anything else to skip: ").strip().upper()

if response == "YES":
    should_publish = True
    print("\nPublishing in progress...\n")
else:
    print("\nSkipped – no comment was posted.\n")

if should_publish:
    # Jira API endpoint for adding comments
    jira_url = f"{os.environ['JIRA_INSTANCE_URL']}/rest/api/3/issue/{ISSUE_KEY}/comment"
    
    # Authentication
    # auth = (os.environ['JIRA_EMAIL'], os.environ['JIRA_API_TOKEN'])

    auth = ("haridurai1230@gmail.com", "ATATT3xFfGF09wnqlrQAR9iwNHNxxoxgbfNSkltPbwCwZ0p7IjRuSEfhcOzBFEYlxoTWJyC0HXIwaukvGGkPP13wW6VDnUvaWWd5sWIHYAdEbdZujOKZ2G4tXxyKC7jUpiNcTAQaKRfxSgbRtc8UpqM-2wN28E8K-qREnLCHQ0F2l7hXhjNksB0=069F4272")
    
    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json"
    }
    
    # Format the test cases for Jira comment
    # Convert test_cases to proper Atlassian Document Format (ADF)
    comment_data = {
        "body": {
            "type": "doc",
            "version": 1,
            "content": [
                {
                    "type": "paragraph",
                    "content": [
                        {
                            "type": "text",
                            "text": "Test Cases Generated By AI",
                            "marks": [{"type": "strong"}]
                        }
                    ]
                },
                {
                    "type": "paragraph",
                    "content": [
                        {
                            "type": "text",
                            "text": f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
                        }
                    ]
                },
                {
                    "type": "rule"
                },
                {
                    "type": "codeBlock",
                    "attrs": {
                        "language": "text"
                    },
                    "content": [
                        {
                            "type": "text",
                            "text": test_cases
                        }
                    ]
                }
            ]
        }
    }
    
    try:
        # Make the API call with SSL verification disabled
        result = requests.post(
            jira_url, 
            json=comment_data, 
            headers=headers, 
            auth=auth,
            verify=False  # Disable SSL verification
        )
        
        # Check response
        if result.status_code == 201:
            comment_response = result.json()
            comment_id = comment_response.get('id')
            comment_url = f"{os.environ['JIRA_INSTANCE_URL']}/browse/{ISSUE_KEY}?focusedCommentId={comment_id}#comment-{comment_id}"
            
            display(Markdown(
                f"**✅ Test cases successfully posted!**\n\n"
                f"→ [View comment in Jira]({comment_url})"
            ))
        else:
            print(f"❌ Error posting comment: {result.status_code}")
            print(f"Response: {result.text}")
            
    except Exception as e:
        print(f"❌ Error posting comment: {str(e)}")
        
else:
    print("(dry run completed – nothing was sent to Jira)")


Do you want to publish these test cases to Jira?


Publishing in progress...



**✅ Test cases successfully posted!**

→ [View comment in Jira](https://haridurai1230.atlassian.net/browse/MBA-8?focusedCommentId=10235#comment-10235)